In [ ]:
!pip install -q transformers==4.46.3 trl==0.12.2 peft==0.13.2 accelerate==1.0.1
!pip install -q -U bitsandbytes

import os
os.kill(os.getpid(), 9)   # restart zaroori hai bitsandbytes upgrade ke baad

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.9/330.9 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 68.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.24.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 22.9 MB/s eta 0:00:00


In [1]:
!pip install -q transformers==4.46.3 trl==0.12.2 peft==0.13.2 accelerate==1.0.1

import torch, gc, os, time
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from peft import PeftModel

BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B"
ADAPTER_PATH = "nooruiit-864/qwen2.5-1.5b-base-ai-safety-domain-lora"  # apna actual Day 30 adapter path confirm karo
MERGED_MODEL_DIR = "/content/merged_model_day31"
os.makedirs(MERGED_MODEL_DIR, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Torch: 2.11.0+cu128 | CUDA: True


In [2]:
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME, torch_dtype=torch.float16, device_map="auto",
)
model_with_adapter = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
merged_model = model_with_adapter.merge_and_unload()

merged_model.save_pretrained(MERGED_MODEL_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_MODEL_DIR)

print("Merged model saved to:", MERGED_MODEL_DIR)
print("Files:", os.listdir(MERGED_MODEL_DIR))

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/720 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/73.9M [00:00<?, ?B/s]

Merged model saved to: /content/merged_model_day31
Files: ['vocab.json', 'generation_config.json', 'config.json', 'tokenizer.json', 'special_tokens_map.json', 'model.safetensors', 'added_tokens.json', 'merges.txt', 'tokenizer_config.json']


In [ ]:
test_pipe = pipeline("text-generation", model=merged_model, tokenizer=tokenizer)
test_pipe.model.generation_config.pad_token_id = tokenizer.eos_token_id
output = test_pipe("Deepfake technology has made it increasingly difficult to", max_new_tokens=60, do_sample=True, temperature=0.7)
print(output[0]["generated_text"])

Deepfake technology has made it increasingly difficult to distinguish between real and fake videos. As a result, a new term, deepfakes, has emerged to describe the process of using artificial intelligence to create synthetic media that appears as real as possible.


In [ ]:
del base_model, model_with_adapter, test_pipe
gc.collect()
torch.cuda.empty_cache()

def get_dir_size_mb(path):
    total = 0
    for dirpath, _, filenames in os.walk(path):
        for f in filenames:
            total += os.path.getsize(os.path.join(dirpath, f))
    return total / (1024 * 1024)

merged_fp16_size = get_dir_size_mb(MERGED_MODEL_DIR)
print(f"Merged model (FP16) disk size: {merged_fp16_size:.1f} MB")

Merged model (FP16) disk size: 2959.6 MB


**Quantized model load + benchmark**

In [1]:
def benchmark_model(model, tokenizer, prompt, max_new_tokens=60, num_runs=3):
    model.eval()
    model.generation_config.pad_token_id = tokenizer.eos_token_id
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        _ = model.generate(**inputs, max_new_tokens=10, do_sample=False)
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    times = []
    for _ in range(num_runs):
        start = time.time()
        with torch.no_grad():
            output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        torch.cuda.synchronize()
        times.append(time.time() - start)
    avg_time = sum(times) / len(times)
    tokens_generated = output.shape[1] - inputs["input_ids"].shape[1]
    return {
        "avg_latency_sec": round(avg_time, 3),
        "tokens_per_sec": round(tokens_generated / avg_time, 2),
        "peak_gpu_mem_gb": round(torch.cuda.max_memory_allocated() / 1e9, 3),
    }

BENCH_PROMPT = "Deepfake technology has made it increasingly difficult to"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)
quantized_model = AutoModelForCausalLM.from_pretrained(MERGED_MODEL_DIR, quantization_config=bnb_config, device_map="auto")
weights_only_mb = torch.cuda.memory_allocated() / (1024 * 1024)
print(f"Quantized weights-only GPU memory: {weights_only_mb:.1f} MB")

torch.cuda.reset_peak_memory_stats()
quantized_results = benchmark_model(quantized_model, tokenizer, BENCH_PROMPT)
print("Merged + Quantized (4-bit):", quantized_results)

del quantized_model
gc.collect()
torch.cuda.empty_cache()

NameError: name 'BitsAndBytesConfig' is not defined

### **Base model**

In [ ]:
base_model_only = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME, torch_dtype=torch.float16, device_map="auto",
)
torch.cuda.reset_peak_memory_stats()
base_results = benchmark_model(base_model_only, tokenizer, BENCH_PROMPT)
print("Base model (FP16, no adapter):", base_results)

del base_model_only
gc.collect()
torch.cuda.empty_cache()

Base model (FP16, no adapter): {'avg_latency_sec': 2.479, 'tokens_per_sec': 24.2, 'peak_gpu_mem_gb': 3.103}


In [ ]:
base_for_lora = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME, torch_dtype=torch.float16, device_map="auto",
)
unmerged_lora_model = PeftModel.from_pretrained(base_for_lora, ADAPTER_PATH)
torch.cuda.reset_peak_memory_stats()
unmerged_results = benchmark_model(unmerged_lora_model, tokenizer, BENCH_PROMPT)
print("Unmerged LoRA (base + adapter, FP16):", unmerged_results)

Unmerged LoRA (base + adapter, FP16): {'avg_latency_sec': 3.733, 'tokens_per_sec': 9.64, 'peak_gpu_mem_gb': 3.176}


In [ ]:
from huggingface_hub import login, notebook_login
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

REPO_NAME = "nooruiit-864/qwen2.5-1.5b-base-ai-safety-domain-lora"  # apna naam confirm/adjust karo

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

# Repo create karo (agar already nahi hai)
api.create_repo(repo_id=REPO_NAME, repo_type="model", exist_ok=True, private=False)

# Merged model + tokenizer push karo
merged_model_for_push = AutoModelForCausalLM.from_pretrained(MERGED_MODEL_DIR, torch_dtype=torch.float16)
merged_model_for_push.push_to_hub(REPO_NAME)
tokenizer.push_to_hub(REPO_NAME)

print(f"Pushed to: https://huggingface.co/{REPO_NAME}")

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0n32wj8/model.safetensors:   0%|          | 21.6kB / 3.09GB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpxd0egbut/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Pushed to: https://huggingface.co/nooruiit-864/qwen2.5-1.5b-base-ai-safety-domain-lora


In [ ]:
model_card_content = f"""---
license: apache-2.0
base_model: {BASE_MODEL_NAME}
tags:
- qlora
- lora-merged
- domain-adaptation
- deepfake-detection
- misinformation
- ai-safety
language:
- en
---

# Qwen2.5-1.5B — Deepfake/Misinformation Domain-Adapted (Merged)

## Model Description
This model is a **merged** version of a QLoRA-fine-tuned adapter on top of `{BASE_MODEL_NAME}` (base variant).
The adapter was trained via domain adaptation on an **AI Safety / Deepfake Misinformation** corpus, then merged
into the base weights using `merge_and_unload()` for standalone deployment (no separate adapter needed at inference time).

## Training Lineage
- **Base model**: `{BASE_MODEL_NAME}`
- **Fine-tuning method**: QLoRA (4-bit NF4 quantization during training, double quantization, paged optimizer)
- **Domain corpus**: AI Safety / Deepfake Misinformation
- **Adapter merge**: `peft.PeftModel.merge_and_unload()`
- **Training framework**: `transformers==4.46.3`, `trl==0.12.2`, `peft==0.13.2`

## Recommended Usage: 4-bit Quantized Inference
For memory-efficient deployment, load this model with bitsandbytes NF4 quantization
(the same setup used in our benchmarking):

```python
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    "{REPO_NAME}",
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained("{REPO_NAME}")
```

## Benchmark Results (Colab T4 GPU)

| Model Variant | Weights Size | Avg Latency (s) | Tokens/sec | Peak GPU Memory (GB) |
|---|---|---|---|---|
| Base model (FP16, no adapter) | ~2960 MB | 2.479 | 24.2 | 3.103 |
| Unmerged LoRA (base + adapter, FP16) | ~2960 MB + adapter | 3.733 | 9.64 | 3.176 |
| **Merged + Quantized (4-bit NF4)** | 2959.6 MB → **1099.1 MB** | 2.419 | 14.88 | **1.164** |

**Key takeaway**: The merged+quantized variant reduces peak GPU memory by ~62% versus the base model,
while retaining domain-adapted knowledge and reasonable inference speed — the recommended variant for
memory-constrained deployment.

## Limitations
- Domain adaptation may exhibit mild catastrophic forgetting on general-purpose tasks (observed during Day 30 evaluation).
- Benchmarks were run on a single T4 GPU with `do_sample=False`, batch size 1; results may vary on other hardware/settings.

## Intended Use
Educational / research use for studying domain-adapted LLM behavior on AI safety and misinformation-related text generation.
"""

with open("/content/README.md", "w") as f:
    f.write(model_card_content)

api.upload_file(
    path_or_fileobj="/content/README.md",
    path_in_repo="README.md",
    repo_id=REPO_NAME,
)

print(f"Model card pushed! View at: https://huggingface.co/{REPO_NAME}")

Model card pushed! View at: https://huggingface.co/nooruiit-864/qwen2.5-1.5b-base-ai-safety-domain-lora


# **Day 31 llama-cpp**

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git /content/llama.cpp
!pip install -q -r /content/llama.cpp/requirements.txt

Cloning into '/content/llama.cpp'...
remote: Enumerating objects: 3793, done.
remote: Counting objects: 100% (3793/3793), done.
remote: Compressing objects: 100% (3079/3079), done.
remote: Total 3793 (delta 684), reused 2649 (delta 633), pack-reused 0 (from 0)
Receiving objects: 100% (3793/3793), 35.34 MiB | 7.27 MiB/s, done.
Resolving deltas: 100% (684/684), done.
Updating files: 100% (3445/3445), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 53.3 MB/s eta 0:0

In [ ]:
from huggingface_hub import snapshot_download

MERGED_MODEL_DIR = snapshot_download(
    repo_id="nooruiit-864/qwen2.5-1.5b-base-ai-safety-domain-lora",
    allow_patterns=["*.json", "*.safetensors", "*.txt", "merges.txt", "vocab.json"],  # GGUF file skip karo, wo alag hai
)
print("Model downloaded to:", MERGED_MODEL_DIR)
import os
print("Files:", os.listdir(MERGED_MODEL_DIR))

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/755 [00:00<?, ?B/s]

...

adapter_model.safetensors:   0%|          | 0.00/73.9M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

Model downloaded to: /root/.cache/huggingface/hub/models--nooruiit-864--qwen2.5-1.5b-base-ai-safety-domain-lora/snapshots/273a01d2110abcd45f7d494588fe345e42e04f90
Files: ['merges.txt', 'tokenizer.json', 'special_tokens_map.json', 'vocab.json', 'config.json', 'adapter_config.json', 'generation_config.json', 'model.safetensors', 'adapter_model.safetensors', 'added_tokens.json', 'tokenizer_config.json']


In [ ]:
GGUF_FP16_PATH = "/content/model-fp16.gguf"

!python /content/llama.cpp/convert_hf_to_gguf.py {MERGED_MODEL_DIR} \
    --outfile {GGUF_FP16_PATH} \
    --outtype f16

print("FP16 GGUF size:", os.path.getsize(GGUF_FP16_PATH) / (1024*1024), "MB")

INFO:hf-to-gguf:Loading model: 273a01d2110abcd45f7d494588fe345e42e04f90
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.float16 --> F16, shape = {1536, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.float16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.float16 --> F16, shape = {8960, 1536}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.float16 --> F16, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.float16 --> F16, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.float16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.attn_k.bias,         torch.float16 --> F32, shape = {256}
INFO:hf-to-gguf:blk.0.attn_k.weight,       torch.float16 --> F16, shape = {1536, 256}
INFO:hf-to-gguf:blk.0.attn_outp

In [ ]:
!cmake -B /content/llama.cpp/build -S /content/llama.cpp -DGGML_CUDA=OFF
!cmake --build /content/llama.cpp/build --config Release -j 4 --target llama-quantize

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- llama.cpp version: 0.1.2-dev
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Found Ope

In [ ]:
GGUF_Q4_PATH = "/content/model-Q4_K_M.gguf"

!/content/llama.cpp/build/bin/llama-quantize {GGUF_FP16_PATH} {GGUF_Q4_PATH} Q4_K_M

print("Q4_K_M GGUF size:", os.path.getsize(GGUF_Q4_PATH) / (1024*1024), "MB")

version: 0.1.2-dev (build 1, commit a3b1eff)
built with GNU 11.4.0 for Linux x86_64
llama_quantize: quantizing '/content/model-fp16.gguf' to '/content/model-Q4_K_M.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 25 key-value pairs and 338 tensors from /content/model-fp16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = 273a01d2110abcd45f7d494588fe345e42e04f90
llama_model_loader: - kv   3:                           general.finetune str              = 273a01d2110abcd45f7d494588fe345e42e04f90
llama_model_loader: - kv   4:                         general.size_label str              = 1.5B
llama_model_loader: - kv   5:

In [ ]:
!pip install -q llama-cpp-python

from llama_cpp import Llama

llm = Llama(model_path=GGUF_Q4_PATH, n_ctx=512, verbose=False)
output = llm(
    "Deepfake technology has made it increasingly difficult to",
    max_tokens=60,
    temperature=0.7,
)
print(output["choices"][0]["text"])

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
 tell real from fake. It is also used to spread misinformation, disinformation, and propaganda.


In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.upload_file(
    path_or_fileobj=GGUF_Q4_PATH,
    path_in_repo="model-Q4_K_M.gguf",
    repo_id="nooruiit-864/qwen2.5-1.5b-base-ai-safety-domain-lora",  # same repo jo Day 31 mein use hua: nooruiit-864/qwen2.5-1.5b-base-ai-safety-domain-lora
)
print("GGUF pushed to: https://huggingface.co/nooruiit-864/qwen2.5-1.5b-base-ai-safety-domain-lora/blob/main/model-Q4_K_M.gguf")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /content/model-Q4_K_M.gguf  :   4%|3         | 35.1MB /  986MB            

GGUF pushed to: https://huggingface.co/nooruiit-864/qwen2.5-1.5b-base-ai-safety-domain-lora/blob/main/model-Q4_K_M.gguf
